In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Odhad energie základního stavu Heisenbergova řetězce pomocí VQE

*Odhadovaná spotřeba: dvě minuty na procesoru Eagle r3 (POZNÁMKA: Jedná se pouze o odhad. Tvůj skutečný čas běhu se může lišit.)*

## Pozadí

Tento tutoriál ukazuje, jak sestavit, nasadit a spustit `Qiskit vzor` pro simulaci Heisenbergova řetězce a odhad jeho energie základního stavu. Více informací o `Qiskit vzorech` a o tom, jak lze `Qiskit Serverless` použít k jejich nasazení do cloudu pro spravované provádění, najdeš na naší [stránce s dokumentací pro IBM Quantum&reg; Platform](/guides/serverless).

## Požadavky

Před zahájením tohoto tutoriálu se ujisti, že máš nainstalováno následující:

* Qiskit SDK v1.2 nebo novější, s podporou [vizualizace](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.28 nebo novější (`pip install qiskit-ibm-runtime`)
* Qiskit Serverless (pip install qiskit_serverless)
* IBM Catalog (pip install qiskit-ibm-catalog)

## Nastavení

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
from typing import Sequence


from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives.base import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, Estimator

from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

In [2]:
def visualize_results(results):
    plt.plot(results["cost_history"], lw=2)
    plt.xlabel("Iteration")
    plt.ylabel("Energy")
    plt.show()


def build_callback(
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
    callback_dict: dict,
):
    def callback(current_vector):
        # Keep track of the number of iterations
        callback_dict["iters"] += 1
        # Set the prev_vector to the latest one
        callback_dict["prev_vector"] = current_vector
        # Compute the value of the cost function at the current vector
        current_cost = (
            estimator.run([(ansatz, hamiltonian, [current_vector])])
            .result()[0]
            .data.evs[0]
        )
        callback_dict["cost_history"].append(current_cost)
        # Print to screen on single line
        print(
            "Iters. done: {} [Current cost: {}]".format(
                callback_dict["iters"], current_cost
            ),
            end="\r",
            flush=True,
        )

    return callback

## Krok 1: Mapování klasických vstupů na kvantový problém
*   Vstup: Počet spinů
*   Výstup: Ansatz a hamiltonián modelující Heisenbergův řetězec

Sestrojíme ansatz a hamiltonián, které modelují 10-spinový Heisenbergův řetězec. Nejdříve importujeme některé obecné balíčky a vytvoříme několik pomocných funkcí.

In [3]:
num_spins = 10
ansatz = efficient_su2(num_qubits=num_spins, reps=3)

# Remember to insert your token in the QiskitRuntimeService constructor
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, min_num_qubits=num_spins, simulator=False
)

coupling = backend.target.build_coupling_map()
reduced_coupling = coupling.reduce(list(range(num_spins)))

edge_list = reduced_coupling.graph.edge_list()
ham_list = []

for edge in edge_list:
    ham_list.append(("ZZ", edge, 0.5))
    ham_list.append(("YY", edge, 0.5))
    ham_list.append(("XX", edge, 0.5))

for qubit in reduced_coupling.physical_qubits:
    ham_list.append(("Z", [qubit], np.random.random() * 2 - 1))

hamiltonian = SparsePauliOp.from_sparse_list(ham_list, num_qubits=num_spins)

ansatz.draw("mpl", style="iqp")

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif)

## Krok 2: Optimalizace problému pro spuštění na kvantovém hardwaru
*   Vstup: Abstraktní Circuit, observabilní veličina
*   Výstup: Cílový Circuit a observabilní veličina, optimalizované pro vybrané QPU

Použijeme funkci `generate_preset_pass_manager` z Qiskitu k automatickému vygenerování optimalizační rutiny pro náš Circuit vzhledem k vybranému QPU. Volíme `optimization_level=3`, což poskytuje nejvyšší úroveň optimalizace z přednastavených pass managerů. Také zahrneme plánovací průchody `ALAPScheduleAnalysis` a `PadDynamicalDecoupling` pro potlačení chyb způsobených dekoherencí.

In [4]:
target = backend.target
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
pm.scheduling = PassManager(
    [
        ALAPScheduleAnalysis(durations=target.durations()),
        PadDynamicalDecoupling(
            durations=target.durations(),
            dd_sequence=[XGate(), XGate()],
            pulse_alignment=target.pulse_alignment,
        ),
    ]
)
ansatz_ibm = pm.run(ansatz)
observable_ibm = hamiltonian.apply_layout(ansatz_ibm.layout)
ansatz_ibm.draw("mpl", scale=0.6, style="iqp", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif)

## Krok 3: Spuštění pomocí Qiskit primitiv
*   Vstup: Cílový Circuit a observabilní veličina
*   Výstup: Výsledky optimalizace

Minimalizujeme odhadovanou energii základního stavu systému optimalizací parametrů Circuitu. Použijeme primitivum `Estimator` z Qiskit Runtime k vyhodnocení účelové funkce během optimalizace.

Pro tuto ukázku spustíme výpočet na QPU pomocí primitiv `qiskit-ibm-runtime`. Pro spuštění s primitivy `qiskit` založenými na stavovém vektoru nahraď blok kódu používající primitiva Qiskit IBM Runtime komentovaným blokem.

In [ ]:
# SciPy minimizer routine
def cost_func(
    params: Sequence,
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
) -> float:
    """Ground state energy evaluation."""
    return (
        estimator.run([(ansatz, hamiltonian, [params])])
        .result()[0]
        .data.evs[0]
    )


num_params = ansatz_ibm.num_parameters
params = 2 * np.pi * np.random.random(num_params)

callback_dict = {
    "prev_vector": None,
    "iters": 0,
    "cost_history": [],
}

# Evaluate the problem on a QPU by using Qiskit IBM Runtime
with Session(backend=backend) as session:
    estimator = Estimator()
    callback = build_callback(
        ansatz_ibm, observable_ibm, estimator, callback_dict
    )
    res = minimize(
        cost_func,
        x0=params,
        args=(ansatz_ibm, observable_ibm, estimator),
        callback=callback,
        method="cobyla",
        options={"maxiter": 100},
    )

visualize_results(callback_dict)

## Krok 4: Následné zpracování a vrácení výsledku v požadovaném klasickém formátu
*   Vstup: Odhady energie základního stavu během optimalizace
*   Výstup: Odhadovaná energie základního stavu

In [ ]:
print(f'Estimated ground state energy: {res["fun"]}')

## Nasazení Qiskit vzoru do cloudu
Chceš-li to provést, přesuň výše uvedený zdrojový kód do souboru `./source/heisenberg.py`, obal kód do skriptu, který přijímá vstupy a vrací konečné řešení, a nakonec ho nahraj na vzdálený cluster pomocí třídy `QiskitFunction` z `qiskit-ibm-catalog`. Pokyny pro zadávání externích závislostí, předávání vstupních argumentů a další informace najdeš v [průvodcích Qiskit Serverless](/guides/serverless).

Vstupem vzoru je počet spinů v řetězci. Výstupem je odhad energie základního stavu systému.

In [ ]:
# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()
heisenberg_function = QiskitFunction(
    title="ibm_heisenberg",
    entrypoint="heisenberg.py",
    working_dir="./source/",
)
serverless.upload(heisenberg_function)

### Spuštění Qiskit vzoru jako spravované služby
Jakmile jsme vzor nahráli do cloudu, můžeme ho snadno spustit pomocí klienta `QiskitServerless`.

In [ ]:
# Run the pattern on the remote cluster

ibm_heisenberg = serverless.load("ibm_heisenberg")
job = serverless.run(ibm_heisenberg)
solution = job.result()

print(solution)
print(job.logs())

## Anketa k tutoriálu
Vyplň prosím tuto krátkou anketu a poskytni nám zpětnou vazbu k tomuto tutoriálu. Tvoje postřehy nám pomohou zlepšit nabízený obsah a uživatelský zážitek.

[Odkaz na anketu](https://your.feedback.ibm.com/jfe/form/SV_bfuBwfNeeFBxnim)

> **Note:** This survey is provided by IBM Quantum and relates to the original English content. To give feedback on doQumentation's website, translations, or code execution, please [open a GitHub issue](https://github.com/JanLahmann/doQumentation/issues).